# 17. Normative Multi-Target Workflow (Step-by-Step)

Notebook operativo unico per eseguire e verificare il modeling normativo multi-target.

## Obiettivi
- Addestrare i modelli GAMLSS per entrambi gli indici di atrofia
- Salvare modelli e dataset con z-score
- Verificare copertura, completezza output e metriche principali

## Target inclusi
1. `hippocampus_etiv_ratio` su `all`
2. `hippocampus_etiv_ratio` su `synthseg`
3. `hippocampus_etiv_ratio` su `fastsurfer`
4. `ilv_to_hippocampal_ratio` su `fastsurfer`

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

repo_root = Path('..').resolve()
script_path = repo_root / 'scripts' / 'run_gamlss_multitarget_normative.R'
input_path = repo_root / 'data' / 'combined' / 'normative_modeling_dataset.csv'
out_dir = repo_root / 'models' / 'normative_multitarget'
summary_path = out_dir / 'normative_multitarget_summary.csv'

print('Repo root:', repo_root)
print('Script   :', script_path)
print('Input    :', input_path)
print('Output   :', out_dir)

## 1) Pre-check: esistenza file e colonne richieste

In [ ]:
required_cols = [
    'dataset', 'source_method', 'age', 'sex',
    'hippocampus_etiv_ratio', 'ilv_to_hippocampal_ratio'
]

assert script_path.exists(), f'Script non trovato: {script_path}'
assert input_path.exists(), f'Dataset non trovato: {input_path}'

df = pd.read_csv(input_path)
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f'Colonne mancanti: {missing_cols}'

print('Righe dataset:', len(df))
print('Metodi presenti:', sorted(df['source_method'].dropna().astype(str).str.lower().unique().tolist()))
print('Dataset presenti:', sorted(df['dataset'].dropna().astype(str).unique().tolist()))

for target in ['hippocampus_etiv_ratio', 'ilv_to_hippocampal_ratio']:
    x = df[target]
    print(f'\n{target}:')
    print('  non-null =', int(x.notna().sum()))
    if x.notna().sum() > 0:
        print('  min      =', float(x.min(skipna=True)))
        print('  p01      =', float(x.quantile(0.01)))
        print('  <=0      =', int((x <= 0).sum(skipna=True)))

## 2) Esecuzione training multi-target

Imposta `RUN_TRAINING = True` per lanciare realmente il training.

In [ ]:
RUN_TRAINING = True

cmd = [
    'conda', 'run', '--no-capture-output', '-n', 'normative-model',
    'Rscript', str(script_path), str(input_path), str(out_dir)
]

print('Command:', ' '.join(cmd))

if RUN_TRAINING:
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f'Training fallito con exit code {result.returncode}')
    print('\nTraining completato con successo.')
else:
    print('RUN_TRAINING=False: training non eseguito.')

## 3) Verifica output prodotti

In [ ]:
expected_files = [
    out_dir / 'gamlss_bct_model_hippocampus_etiv_ratio__all.rds',
    out_dir / 'gamlss_bct_model_hippocampus_etiv_ratio__synthseg.rds',
    out_dir / 'gamlss_bct_model_hippocampus_etiv_ratio__fastsurfer.rds',
    out_dir / 'gamlss_bct_model_ilv_to_hippocampal_ratio__fastsurfer.rds',
    out_dir / 'normative_with_zscores_hippocampus_etiv_ratio__all.csv',
    out_dir / 'normative_with_zscores_hippocampus_etiv_ratio__synthseg.csv',
    out_dir / 'normative_with_zscores_hippocampus_etiv_ratio__fastsurfer.csv',
    out_dir / 'normative_with_zscores_ilv_to_hippocampal_ratio__fastsurfer.csv',
    out_dir / 'normative_multitarget_summary.csv',
]

missing = [str(p) for p in expected_files if not p.exists()]
if missing:
    raise FileNotFoundError('Output mancanti:\n' + '\n'.join(missing))

print('Tutti gli output attesi sono presenti.')
print('\nContenuto output directory:')
for p in sorted(out_dir.glob('*')):
    print('-', p.name)

## 4) Riepilogo modelli e diagnostica z-score

In [ ]:
summary_df = pd.read_csv(summary_path)
display(summary_df)

print('\nControllo rapido normalita z-score per run:')
for _, row in summary_df.iterrows():
    print(f"- {row['run_name']}: n={int(row['n'])}, mean_z={row['mean_z']:.4f}, sd_z={row['sd_z']:.4f}, range=[{row['min_z']:.3f}, {row['max_z']:.3f}]")

## 5) Report finale esportabile

In [ ]:
report_path = out_dir / 'normative_multitarget_run_report.txt'
lines = []
lines.append('NORMATIVE MULTI-TARGET RUN REPORT')
lines.append('')
lines.append(f'Input dataset: {input_path}')
lines.append(f'Output dir: {out_dir}')
lines.append(f'Total runs: {len(summary_df)}')
lines.append('')
for _, r in summary_df.iterrows():
    lines.append(f"- {r['run_name']} | n={int(r['n'])} | mean_z={r['mean_z']:.4f} | sd_z={r['sd_z']:.4f} | AIC={r['AIC']:.3f} | BIC={r['SBC']:.3f}")

report_path.write_text('\n'.join(lines), encoding='utf-8')
print('Report salvato in:', report_path)